# 🐍 Python from Scratch — Module 10: APIs, `requests`, and an Intro to `pandas`

### The last stop — data from the internet, and analyzing it

This is the final module in the series, and it deliberately ties almost everything
together: fetching data over the network (`requests`), its structure (JSON — remember
Module 7?), handling network errors (Module 5), and finally a first contact with
`pandas` — the library that's hard to imagine data analysis in Python without.

**Note:** unlike earlier modules, this one needs an internet connection — the cells
that fetch data from the API will only work if your computer has network access.

## Table of Contents

1. [What an API is — a quick reminder](#sec1)
2. [The `requests` library — your first request](#sec2)
3. [Handling network errors](#sec3)
4. [`pandas` — first contact](#sec4)
5. [Loading a CSV into `pandas`](#sec5)
6. [Filtering and sorting](#sec6)
7. [`groupby` — aggregating data](#sec7)
8. [Combining `requests` with `pandas`](#sec8)
9. [Fun fact: where "pandas" and "requests" got their names](#sec9)
10. [Summary of the whole series](#sec10)
11. [Exercises](#sec11)

---

<a id="sec1"></a>
## 1. What an API is — a quick reminder

An **API** (*Application Programming Interface*) is how one program can ask another for
data, or tell it to do something — usually over the internet, by sending a request to a
specific address (URL) and getting back a response, typically in **JSON** format
(remember Module 7 — `import json`?). In this module we'll use a free, public test API:
`jsonplaceholder.typicode.com` — built specifically for learning purposes like this.

<a id="sec2"></a>
## 2. The `requests` library — your first request

`requests` is the most popular package for making HTTP requests in Python (it isn't
part of the standard library - `pip install requests`, though Anaconda usually has it
installed already). `requests.get(url)` sends a GET request and returns a response
object.

In [ ]:
import requests

response = requests.get("https://jsonplaceholder.typicode.com/users/1")

print(response.status_code)   # 200 = success
print(response.headers["Content-Type"])

data = response.json()        # turns the response body (JSON) into a Python dict
print(data)
print(data["name"], data["email"])

> 💡 **HTTP status codes**
>
> 200 means success, but it's worth knowing a few other common codes too: `404` (resource not found), `401`/`403` (unauthorized/forbidden), `500` (server-side error). `response.ok` is a shortcut that returns `True` for codes 200-399.

<a id="sec3"></a>
## 3. Handling network errors

A network request can fail in plenty of ways: no internet, a bad URL, the server not
responding within a reasonable time. It's worth handling this - with exactly the same
tools from Module 5 (`try`/`except`).

In [ ]:
import requests

try:
    response = requests.get(
        "https://jsonplaceholder.typicode.com/users/1",
        timeout=5   # give up after 5 seconds instead of waiting forever
    )
    response.raise_for_status()   # raises an exception if the status is an error (4xx/5xx)
    print(response.json())
except requests.exceptions.Timeout:
    print("The server did not respond in time")
except requests.exceptions.ConnectionError:
    print("No internet connection, or a bad address")
except requests.exceptions.HTTPError as error:
    print(f"The server returned an error: {error}")

> ⚠️ **`raise_for_status()` isn't automatic**
>
> `requests` does NOT raise an exception on its own when the server returns an error (e.g. 404) - `response.status_code` will simply be 404, and `response.ok` will be `False`, but the code carries on normally. You have to explicitly call `raise_for_status()` to turn a bad status into a Python exception you can catch with `except`.

<a id="sec4"></a>
## 4. `pandas` — first contact

`pandas` is a library for working with tabular data - its central object is the
**DataFrame**, something like a spreadsheet living inside Python: rows, columns, each
column with its own data type.

In [ ]:
import pandas as pd

products = [
    {"name": "Bread", "price": 4.5, "qty": 20},
    {"name": "Milk", "price": 3.2, "qty": 15},
    {"name": "Eggs", "price": 12.0, "qty": 8},
]

df = pd.DataFrame(products)
print(df)
print("---")
print(df.head(2))       # the first 2 rows
print("---")
print(df.describe())    # basic statistics for the numeric columns
print("---")
print(df["price"])       # access a single column
print(df["price"].mean())   # average price

> 💡 **Fun fact**
>
> `pandas` is almost always imported under the alias `pd` (`import pandas as pd`) - the convention is so strong that pretty much every pandas snippet in the world uses it. Similarly, `numpy` is imported as `np`. Worth getting used to right away.

<a id="sec5"></a>
## 5. Loading a CSV into `pandas`

`pandas` has a built-in function for loading a CSV directly into a DataFrame - much
more convenient than manually looping over `csv.DictReader` like in Module 8.

In [ ]:
import pandas as pd
import csv

# First, let's create a CSV file (as in Module 8):
with open("products.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["name", "price", "qty"])
    writer.writerow(["Bread", 4.5, 20])
    writer.writerow(["Milk", 3.2, 15])
    writer.writerow(["Eggs", 12.0, 8])
    writer.writerow(["Coffee", 25.0, 5])

df = pd.read_csv("products.csv")
print(df)
print(df.dtypes)   # each column's data type, detected automatically

<a id="sec6"></a>
## 6. Filtering and sorting

Filtering rows in `pandas` looks different from an `if` inside a loop - it uses a
**boolean mask**: a condition that evaluates to `True`/`False` for every row, then used
as an "index" on the DataFrame.

In [ ]:
import pandas as pd

df = pd.read_csv("products.csv")

expensive = df[df["price"] > 5]           # boolean mask - only rows with price > 5
print(expensive)

sorted_df = df.sort_values("price", ascending=False)   # sort descending
print(sorted_df)

print(df[df["price"] > 5]["name"])    # filter + select a single column

<a id="sec7"></a>
## 7. `groupby` — aggregating data

`groupby` groups rows by a column's value and lets you compute an aggregate statistic
for every group at once - the equivalent of a manual counter dictionary from Module 3,
just in a single line.

In [ ]:
import pandas as pd

orders = pd.DataFrame([
    {"customer": "Kamil", "total": 50},
    {"customer": "Ania", "total": 30},
    {"customer": "Kamil", "total": 20},
    {"customer": "Tomek", "total": 100},
    {"customer": "Ania", "total": 15},
])

total_per_customer = orders.groupby("customer")["total"].sum()
print(total_per_customer)

print("---")
print(orders.groupby("customer")["total"].agg(["sum", "mean", "count"]))

<a id="sec8"></a>
## 8. Combining `requests` with `pandas`

A typical workflow: fetch data from an API (a list of dictionaries in JSON), build a
DataFrame from it, and keep analyzing with `pandas` tools.

In [ ]:
import requests
import pandas as pd

response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=5)
response.raise_for_status()
users = response.json()   # a list of dictionaries

df = pd.DataFrame(users)
print(df.columns.tolist())         # what columns came back from the API
print(df[["name", "email", "phone"]].head())

<a id="sec9"></a>
## 9. Fun fact: where "pandas" and "requests" got their names

Two completely different naming stories.

> 💡 **Fun fact**
>
> The name `pandas` doesn't come from the animal - it's short for «panel data», an econometrics term for multi-dimensional datasets. `requests`, on the other hand, has the unofficial motto «HTTP for Humans» - it was created because the built-in alternative (`urllib`) was widely considered unreadable and awkward to use.

<a id="sec10"></a>
## 10. Summary of the whole series

By now it should be clear:

- how to send GET requests with `requests` and read a JSON response,
- how to handle network errors (`timeout`, `ConnectionError`, `raise_for_status()`),
- how to build a DataFrame from a list of dictionaries or a CSV file,
- the basic operations: `.head()`, `.describe()`, filtering with a boolean mask,
  `.sort_values()`,
- how to aggregate data with `.groupby()`.

This wraps up the ten-module series: from `print("Hello, world!")` to fetching and
analyzing data from the internet. Along the way you covered everything needed to write
real programs: variables, control flow, collections, functions, error and file
handling, object-oriented programming, organizing code into modules, text processing,
testing, and finally - working with data from the outside world.

<a id="sec11"></a>
## 11. Exercises

The final set of exercises in the whole series - the last task is a small project
tying several modules together at once.

> 📝 **Exercise 1: A single user from the API**
>
> Fetch user id `3` from `https://jsonplaceholder.typicode.com/users/3` and print their name, company name (`company.name` in the returned dictionary), and city (`address.city`).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import requests

response = requests.get("https://jsonplaceholder.typicode.com/users/3", timeout=5)
user = response.json()

print(user["name"])
print(user["company"]["name"])
print(user["address"]["city"])
```
</details>

> 📝 **Exercise 2: Counting a given user's posts**
>
> Fetch all posts from `https://jsonplaceholder.typicode.com/posts` (a list of dictionaries, each with a `userId` key). Count how many posts belong to `userId == 1`, without pandas - a plain `for` loop.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=5)
posts = response.json()

count = 0
for post in posts:
    if post["userId"] == 1:
        count += 1

print(f"User 1 has {count} posts")
```
</details>

> 📝 **Exercise 3: A safe request with full error handling**
>
> Write a function `fetch_data(url)` that makes a GET request with `timeout=5`, calls `raise_for_status()`, and inside a `try`/`except` block handles `Timeout`, `ConnectionError`, and `HTTPError`, returning `None` and printing a message for each case, otherwise returning `response.json()`. Test it on a valid address and on a deliberately broken one (e.g. `https://jsonplaceholder.typicode.com/users/99999`).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import requests

def fetch_data(url):
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print("The server did not respond in time")
        return None
    except requests.exceptions.ConnectionError:
        print("No internet connection")
        return None
    except requests.exceptions.HTTPError as error:
        print(f"HTTP error: {error}")
        return None

print(fetch_data("https://jsonplaceholder.typicode.com/users/1"))
print(fetch_data("https://jsonplaceholder.typicode.com/users/99999"))
```

Hint: `/users/99999` returns an empty `{}` object (not a 404 error) on this particular test API - a good chance to notice that different APIs signal «not found» differently, so it's always worth checking the docs for whatever specific API you're using.
</details>

> 📝 **Exercise 4: A DataFrame from the inventory (Module 3/6)**
>
> Go back to the product list from Module 3/6 (`name`, `price`, `qty`). Build a DataFrame from it, add a new column `value = price * qty` (assigning to a new column works like assigning to a dictionary key: `df['value'] = ...`), and print `df.describe()`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import pandas as pd

products = [
    {"name": "Bread", "price": 4.5, "qty": 20},
    {"name": "Milk", "price": 3.2, "qty": 15},
    {"name": "Eggs", "price": 12.0, "qty": 8},
]

df = pd.DataFrame(products)
df["value"] = df["price"] * df["qty"]
print(df)
print(df.describe())
```
</details>

> 📝 **Exercise 5: Filtering and sorting a DataFrame**
>
> Using the DataFrame from the previous exercise, print only the products with `value > 50`, sorted descending by `value`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import pandas as pd

products = [
    {"name": "Bread", "price": 4.5, "qty": 20},
    {"name": "Milk", "price": 3.2, "qty": 15},
    {"name": "Eggs", "price": 12.0, "qty": 8},
]

df = pd.DataFrame(products)
df["value"] = df["price"] * df["qty"]

result = df[df["value"] > 50].sort_values("value", ascending=False)
print(result)
```
</details>

> 📝 **Exercise 6: `groupby` on orders**
>
> Given a list of orders (dictionaries with `customer` and `total` keys, a few customers repeating), build a DataFrame and compute, with `.groupby()`, the total order amount per customer, sorted descending.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import pandas as pd

orders = pd.DataFrame([
    {"customer": "Kamil", "total": 50},
    {"customer": "Ania", "total": 30},
    {"customer": "Kamil", "total": 20},
    {"customer": "Tomek", "total": 100},
    {"customer": "Ania", "total": 15},
])

result = orders.groupby("customer")["total"].sum().sort_values(ascending=False)
print(result)
```
</details>

> 📝 **Exercise 7: API + pandas together**
>
> Fetch the list of users from `https://jsonplaceholder.typicode.com/users`, and build a DataFrame from it containing only the columns `name`, `email`, and `city` (hint: `city` needs to be pulled out manually from the nested `address` BEFORE building the DataFrame, e.g. a list comprehension building a new list of flat dictionaries).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import requests
import pandas as pd

response = requests.get("https://jsonplaceholder.typicode.com/users", timeout=5)
users = response.json()

flattened = [
    {"name": u["name"], "email": u["email"], "city": u["address"]["city"]}
    for u in users
]

df = pd.DataFrame(flattened)
print(df)
```
</details>

> 🔥 **Exercise 8 (challenge): A mini API-to-CSV report**
>
> Fetch all posts from `https://jsonplaceholder.typicode.com/posts` inside a `try`/`except` block (handle network errors as in exercise 3). Build a DataFrame, compute the number of posts per user with `.groupby('userId')` (hint: `.size()` instead of `.sum()`), sort descending by post count, and save the result to a file `post_report.csv` with `.to_csv()`. Finally, read that file back with `pd.read_csv()` to confirm the save worked.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import requests
import pandas as pd

try:
    response = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=5)
    response.raise_for_status()
    posts = response.json()
except requests.exceptions.RequestException as error:
    print(f"Error fetching data: {error}")
    posts = []

if posts:
    df = pd.DataFrame(posts)
    report = df.groupby("userId").size().sort_values(ascending=False)
    report.name = "post_count"
    report.to_csv("post_report.csv")

    loaded = pd.read_csv("post_report.csv")
    print(loaded)
```

Hint: `requests.exceptions.RequestException` is the common base class for `Timeout`, `ConnectionError`, and `HTTPError` - catching it with a single `except` is a convenient shortcut when you don't need a different message for each error type, and just want to safely handle «anything that went wrong with the network».
</details>

---

### That's the end of this series

Ten modules, from your very first `print()` to fetching and analyzing data from the
internet. The natural next steps from here, depending on where you want to go: charts
and data visualization (`matplotlib`, `seaborn`), deeper `pandas` (joining tables,
cleaning messy data), building simple web applications (`Flask`, `FastAPI`), or machine
learning (`scikit-learn`). Congratulations on making it through the whole path — that's
a solid foundation to build on.